In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
warnings.filterwarnings('ignore')

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
NNRUNS    = CONFIGS['experiments']['nn']['runs']
EQUATIONS = CONFIGS['experiments']['sr']['optimizedeqs']
LABELS    = {**{name:run['description'] for name,run in NNRUNS.items()},**{name:eq['description'] for name,eq in EQUATIONS.items()}}
SPLIT     = 'test'

In [ ]:
def calc_seed_mean(ytrue,ypred):
    scores = []
    for seed in (ypred['seed'].values if 'seed' in ypred.dims else [None]):
        seedtrue,seedpred = xr.align(ytrue,(ypred if seed is None else ypred.sel(seed=seed)).squeeze(),join='inner')
        if int(seedpred.count())==0:
            continue
        sqerr = (seedtrue.squeeze()-seedpred)**2
        scores.append((float(1-sqerr.sum()/((seedtrue-seedtrue.mean())**2).sum()),float(sqerr.mean())))
    return tuple(np.mean(scores,axis=0)) if scores else (np.nan,np.nan)

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds['tp'].load()
years = np.unique(truetp['time'].dt.year.values)
rows  = []
for name,label in LABELS.items():
    path = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(path):
        print(f'Skipping {label}, missing predictions')
        continue
    with xr.open_dataset(path) as ds:
        predtp = ds['tp'].load()
    row = dict(model=label,all=calc_seed_mean(truetp,predtp)[0])
    for year in years:
        row[int(year)] = calc_seed_mean(truetp.sel(time=truetp['time'].dt.year==year),predtp.sel(time=predtp['time'].dt.year==year))[0]
    rows.append(row)
table = pd.DataFrame(rows).set_index('model').sort_values('all')
print('Test R2 by year:')
print(table.round(3).to_string())

In [ ]:
ranks = table.rank(ascending=False).astype(int)
print('Rank by year (1 = highest R2):')
print(ranks.to_string())
reference = ranks['all']
for year in years:
    swapped = ranks.index[ranks[int(year)]!=reference].tolist()
    print(f"{int(year)}: {'same ranking as all test years' if not swapped else 'rank differs for '+', '.join(swapped)}")